In [1]:
import os
import numpy as np
import pandas as pd

import math
import numba
from numba import cuda

In [2]:
@cuda.jit
def calculate_clusters(data, centroids, assign):
    '''thread calculates its distance from from each centroid and assigns centroid for each point'''

    i = cuda.grid(1) # thread index
    row = data.shape[0]
    dim = data.shape[1]
    clusters = centroids.shape[0]

    # shared memory for centroids
    th_id = cuda.threadIdx.x
    shared_centroids = cuda.shared.array(shape=(max_k, max_d), dtype=np.float64)

    # splitting among threads
    if th_id < (clusters * dim):

        # using width
        k = th_id // dim
        j = th_id % dim
        shared_centroids[k, j] = centroids[k, j]
        
    # wait for all threads to finish
    cuda.syncthreads()

    if i >= row:
        return
        
    best_cluster = 0
    best_distance = math.inf


    #########################################
    for k in range(clusters):
        
        distance = 0.0
        for j in range(dim):
            diff = data[i, j]-shared_centroids[k, j] # dist from centroid
            distance += diff * diff

        if distance < best_distance:
            best_distance = distance
            best_cluster = k

    assign[i] = best_cluster

In [3]:
@cuda.jit
def cal_centroid_sums(data, assignments, centroid_sums):
    '''give each block one dim of a cluster & get sum of all points of a cluster'''

    
    # thread index, block idx and no of threads in a block
    th_id = cuda.threadIdx.x
    block_id = cuda.blockIdx.x
    block_size = cuda.blockDim.x

    # print(th_id)
    # if th_id == 4:
    #     print('666666',th_id, block_id, block_size)

    
    row = data.shape[0]
    dim = data.shape[1]

    # map blocks with k j
    k = block_id // dim # controid and dimension
    j = block_id % dim

    if k >= centroid_sums.shape[0]:
        return

    # thread in the same block shares memory calculation
    shared_sum = cuda.shared.array(shape=THREADS_PER_BLOCK, dtype=np.float64)

    
    # single thread sum
    local_sum = 0.0   
    i = th_id

    #########################################
    while i < row:
        if assignments[i] == k: # data point belonging to a prticular cluster
            local_sum += data[i, j]
        i += block_size  # distribute among each thread

    # storing partial result in shared memory
    shared_sum[th_id] = local_sum

    cuda.syncthreads() # wait for threads calculations

    
    #### parallel reduction
    stride = block_size // 2


    #########################################
    while stride > 0:
        if th_id < stride:

            shared_sum[th_id] += shared_sum[th_id + stride]

        cuda.syncthreads()
        stride //= 2   #pr


    # Thread 0 contains the final sum
    if th_id == 0:
        centroid_sums[k, j] = shared_sum[0]
        

In [4]:
@cuda.jit
def cal_cluster_count(assignments, cluster_counts):
    '''count points in a cluster'''

    th_id = cuda.threadIdx.x
    block_size = cuda.blockDim.x

    block_id = cuda.blockIdx.x
    N = assignments.shape[0]
    K = cluster_counts.shape[0]

    # block for each cluster
    if block_id >= K:
        return

    shared_count = cuda.shared.array(shape=THREADS_PER_BLOCK, dtype=np.int32)

    # each thread counts some of the points belonging to cluster k
    local_count = 0
    i = th_id

    #############################################3
    # count points of one cluster ata time
    while i < N:
        if assignments[i] == block_id:
            local_count += 1
        i += block_size # distribute

    shared_count[th_id] = local_count
    cuda.syncthreads()


    #### parll reduction
    stride = block_size // 2

    while stride > 0:
        if th_id < stride:
            shared_count[th_id] += shared_count[th_id + stride]

        cuda.syncthreads()
        stride //= 2
    
    if th_id == 0:
        cluster_counts[block_id] = shared_count[0]

In [5]:
@cuda.jit
def update_centroids(centroid_sums, cluster_counts, centroids):
    '''updated centroid with mean'''
    
    block_id = cuda.blockIdx.x
    th_id = cuda.threadIdx.x

    K = centroids.shape[0]
    D = centroids.shape[1]

    
    if block_id >= K or th_id >= D:
        return

    count = cluster_counts[block_id]

    if count > 0:
        # avg
        centroids[block_id, th_id] = (centroid_sums[block_id, th_id] / count)

In [6]:
import time
import pandas as pd

# df = pd.read_csv("./rows10000/synthetic_data_10000_rows_11_cols_3_clusters.csv")
# df.head()

In [7]:
def run_kmeans(df_data, N, D, K, max_iter=400, tol=1e-4):
    print(f"--- Running K-Means (N={N}, D={D}, K={K}) ---")
    
    # initialize data and centroids
    np.random.seed(42)
    h_data = df_data.to_numpy()
    
    
    # randomly pick K points from data as initial centroids
    initial_indices = np.random.choice(N, K, replace=False) # array of length dim
    h_centroids = h_data[initial_indices].copy()
    # assign all point to cluster 0
    h_assign = np.zeros(N, dtype=np.int32)

    
    # move data to Gpu
    d_data = cuda.to_device(h_data) # entire data
    d_centroids = cuda.to_device(h_centroids) # random K no. points chosen as centroid
    d_assign = cuda.to_device(h_assign) # zeros

    
    start_time = time.time()
    
    iteration = 0
    for iteration in range(max_iter):
        old_centroids = h_centroids.copy()
        
        # reset sums and counts for this iteration
        d_centroid_sums = cuda.to_device(np.zeros((K, D), dtype=np.float64))
        d_cluster_counts = cuda.to_device(np.zeros(K, dtype=np.int32))

        ##assign each point nearest centroid
        calculate_clusters[blocks_per_grid_points, THREADS_PER_BLOCK](d_data, d_centroids, d_assign)
        cuda.synchronize()

        ## parallel Reduction for sums of points
        cal_centroid_sums[K*D, THREADS_PER_BLOCK](d_data, d_assign, d_centroid_sums)
        cuda.synchronize()

        ## parallel Reduction for count of points
        cal_cluster_count[K, THREADS_PER_BLOCK](d_assign, d_cluster_counts)
        cuda.synchronize()

        ## update centroids
        update_centroids[K, D](d_centroid_sums, d_cluster_counts, d_centroids)
        cuda.synchronize()

        
        # give value to host cpu
        d_centroids.copy_to_host(h_centroids)
        # maximum centroid shift
        cent_shift = np.max(np.sqrt(np.sum((h_centroids - old_centroids)**2, axis=1)))

        
        if cent_shift < tol:
            print(f"\nConverged early at iteration {iteration + 1} (Shift: {cent_shift:.6f} < {tol})")
            break

    # Download final assignments
    d_assign.copy_to_host(h_assign)
    
    exec_time = time.time() - start_time
    
    print(f"Execution Time: {exec_time:.4f} seconds")
    print(f"Total Iterations: {iteration + 1}")
    
    print("\n\nFinal Centroids:\n", h_centroids)
    print("-" * 40)
    
    return h_assign, h_centroids, iteration + 1, exec_time

In [8]:

if __name__ == "__main__":

    # shared memory size
    max_k = 64
    max_d = 32
    
    
    df = pd.read_csv("./rows1000000/synthetic_data_1000000_rows_10_cols_17_clusters.csv")
    d_dims = ['col_1', 'col_2', 'col_3', 'col_4', 'col_5', 'col_6', 'col_7', 'col_8', 'col_9']
    df_data = df[d_dims]
    
    
    D1 = len(d_dims)
    N1 = len(df)

    THREADS_PER_BLOCK = 256
    blocks_per_grid_points = math.ceil(N1 / THREADS_PER_BLOCK)

    
    # test
    run_kmeans(df_data, N=N1, D=D1, K=17)
    

--- Running K-Means (N=1000000, D=9, K=17) ---


/home/cvpro/anaconda3/envs/trnsf/lib/python3.10/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 17 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/cvpro/anaconda3/envs/trnsf/lib/python3.10/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 17 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))



Converged early at iteration 224 (Shift: 0.000000 < 0.0001)
Execution Time: 3.9301 seconds
Total Iterations: 224


Final Centroids:
 [[262.86986918  59.01306623  13.77348434 546.39795812  87.59752349
  917.59997623 539.13210943 281.55671923 550.00067747]
 [173.04745569 189.46681977 786.13550128 663.43025445 884.20665528
  323.4400164  228.29982662 463.52767514  38.92410262]
 [784.94061563 328.05282968 218.00821091 647.60687964  50.36594384
  121.95301823 -92.98598752 765.45831792 628.60889986]
 [  6.05406033 892.84238038 401.33433543 122.58752663 236.52928068
  537.10031268  94.43551432 842.2787986  734.37742857]
 [675.65587729 758.83469669 404.81774966 525.64291873  53.76828884
   26.02320267 635.31319052 418.21419524 521.77734057]
 [ 42.9428965  423.27780416 149.64093303 636.7685966  380.85309556
  815.95502621 670.33120235 243.58994082 815.47157209]
 [751.36504391 382.75866488 844.43384901 667.11167169   3.59445851
  973.14546293 737.2639939  764.70758298  40.93542982]
 [740.460204

-----